# Text Feature Engineering Assignment

## Problem Statement

Building a Text Processing Pipeline for analyzing user-generated text data like reviews and comments, converting them into numerical features for ML models. Implement One Hot Encoding, Bag of Words, and TF-IDF on real-world scraped data.

## Dataset Collection

Scrape at least 100 product reviews of shoes from Flipkart or Amazon using BeautifulSoup or Selenium. Store in CSV with 'review_text' column.

### Step 1: Scrape 100 Product Reviews

Use BeautifulSoup or Selenium to scrape at least 100 shoe reviews from Amazon or Flipkart. Save the results to a local CSV file named `shoe_reviews.csv` with a `review_text` column.

> Note: In this notebook, the dataset is loaded from the local `shoe_reviews.csv` file to avoid repeated scraping and to keep the analysis reproducible.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# Example scraping function for Amazon reviews using BeautifulSoup
# Note: Amazon frequently changes its markup and blocks scraping,
# so this is a template. In this notebook, we load the saved CSV instead.

def scrape_amazon_reviews(product_url, num_reviews=100):
    reviews = []
    page = 1
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    while len(reviews) < num_reviews:
        url = f"{product_url}&pageNumber={page}"
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')

        # Amazon review elements may vary; this is an example selector.
        review_elements = soup.find_all('span', {'data-hook': 'review-body'})
        for review in review_elements:
            text = review.get_text(strip=True)
            if text:
                reviews.append(text)
            if len(reviews) >= num_reviews:
                break

        page += 1
        time.sleep(1)

    return reviews[:num_reviews]

# Example usage (commented out for reproducibility):
# product_url = 'https://www.amazon.com/product-reviews/PRODUCT_ID'
# reviews = scrape_amazon_reviews(product_url, num_reviews=100)
# pd.DataFrame({'review_text': reviews}).to_csv('shoe_reviews.csv', index=False)

# Load the saved CSV file for further analysis
csv_path = 'shoe_reviews.csv'
df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} reviews from {csv_path}")
print(df.head())

In [1]:
import pandas as pd

# Load existing local review dataset
csv_path = 'shoe_reviews.csv'
df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} reviews from {csv_path}")
print(df.head())

Loaded 120 reviews from shoe_reviews.csv
                                         review_text
0  Very comfortable shoes with good arch support ...
1  The shoes look great but feel a bit tight arou...
2  I love the style, color is exactly as pictured...
3  These shoes are not durable; the sole started ...
4  Good value for the price, easy to wear all day...


## Preprocessing

Convert text to lowercase, tokenization, remove punctuation, optional stopwords and lemmatization.

In [15]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import string

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()


def lowercase_text(text):
    return text.lower()


def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))


def tokenize(text):
    return text.split()


def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]


def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]


def preprocess_text(text):
    text_lower = lowercase_text(text)
    text_no_punct = remove_punctuation(text_lower)
    tokens = tokenize(text_no_punct)
    tokens_no_stop = remove_stopwords(tokens)
    tokens_lemma = lemmatize_tokens(tokens_no_stop)
    return ' '.join(tokens_lemma)

# Show step-by-step processing for the first review
sample_text = df['review_text'].iloc[0]
print('Original text:')
print(sample_text)
print('\nLowercase:')
print(lowercase_text(sample_text))
print('\nRemove punctuation:')
print(remove_punctuation(lowercase_text(sample_text)))
print('\nTokenization:')
print(tokenize(remove_punctuation(lowercase_text(sample_text))))
print('\nRemove stopwords:')
print(remove_stopwords(tokenize(remove_punctuation(lowercase_text(sample_text)))))
print('\nLemmatization:')
print(lemmatize_tokens(remove_stopwords(tokenize(remove_punctuation(lowercase_text(sample_text))))))

# Apply preprocessing to all reviews
df['processed_text'] = df['review_text'].apply(preprocess_text)
print('\nProcessed text sample:')
print(df[['review_text', 'processed_text']].head())

Original text:
Very comfortable shoes with good arch support and cushy cushioning.

Lowercase:
very comfortable shoes with good arch support and cushy cushioning.

Remove punctuation:
very comfortable shoes with good arch support and cushy cushioning

Tokenization:
['very', 'comfortable', 'shoes', 'with', 'good', 'arch', 'support', 'and', 'cushy', 'cushioning']

Remove stopwords:
['comfortable', 'shoes', 'good', 'arch', 'support', 'cushy', 'cushioning']

Lemmatization:
['comfortable', 'shoe', 'good', 'arch', 'support', 'cushy', 'cushioning']

Processed text sample:
                                         review_text  \
0  Very comfortable shoes with good arch support ...   
1  The shoes look great but feel a bit tight arou...   
2  I love the style, color is exactly as pictured...   
3  These shoes are not durable; the sole started ...   
4  Good value for the price, easy to wear all day...   

                                      processed_text  
0  comfortable shoe good arch suppor

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/Ashish_1/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/Ashish_1/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## Vocabulary Creation

Build vocabulary using sklearn. Print vocabulary size and analyze top frequent words.

In [16]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['processed_text'])
vocab = vectorizer.get_feature_names_out()
print(f"Vocabulary size: {len(vocab)}")

# Top frequent words
word_freq = X.sum(axis=0).A1
word_freq_dict = dict(zip(vocab, word_freq))
sorted_words = sorted(word_freq_dict.items(), key=lambda x: x[1], reverse=True)
print("Top 10 frequent words:")
for word, freq in sorted_words[:10]:
    print(f"{word}: {freq}")

Vocabulary size: 242
Top 10 frequent words:
shoe: 59
comfortable: 21
feel: 17
good: 15
foot: 12
cushioning: 11
fit: 11
sole: 11
wear: 11
great: 9


## Feature Engineering

Implement One Hot Encoding (document-level), Bag of Words, and TF-IDF.

In [17]:
# One Hot Encoding (binary presence)
from sklearn.feature_extraction.text import CountVectorizer

ohe_vectorizer = CountVectorizer(binary=True)
ohe_matrix = ohe_vectorizer.fit_transform(df['processed_text'])
print("One Hot Encoding shape:", ohe_matrix.shape)

# Bag of Words (already have from vocab)
bow_matrix = X
print("Bag of Words shape:", bow_matrix.shape)

# TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(df['processed_text'])
print("TF-IDF shape:", tfidf_matrix.shape)

One Hot Encoding shape: (120, 242)
Bag of Words shape: (120, 242)
TF-IDF shape: (120, 242)


## Comparison Analysis

Create a comparison table for OHE, BoW, and TF-IDF. Explain TF-IDF importance and weighting.

In [11]:
import pandas as pd

comparison = pd.DataFrame({
    'Method': ['One Hot Encoding', 'Bag of Words', 'TF-IDF'],
    'Description': ['Binary vector indicating presence of words', 'Count of word occurrences', 'TF-IDF score weighting importance'],
    'Shape': [ohe_matrix.shape, bow_matrix.shape, tfidf_matrix.shape],
    'Values': ['0 or 1', 'Integer counts', 'Float weights']
})
print(comparison)

# Explain TF-IDF
print("\nTF-IDF Explanation:")
print("TF-IDF (Term Frequency-Inverse Document Frequency) gives higher weight to words that are important in a document but rare across the corpus.")
print("Common words like 'the', 'and' get lower weights because they appear in many documents (high IDF).")
print("Rare words that appear frequently in a specific document get higher weights.")

# Example
feature_names = tfidf_vectorizer.get_feature_names_out()
tfidf_scores = tfidf_matrix.sum(axis=0).A1
important_words = sorted(zip(feature_names, tfidf_scores), key=lambda x: x[1], reverse=True)[:10]
print("Top important words in TF-IDF:")
for word, score in important_words:
    print(f"{word}: {score:.2f}")

             Method                                 Description       Shape  \
0  One Hot Encoding  Binary vector indicating presence of words  (120, 242)   
1      Bag of Words                   Count of word occurrences  (120, 242)   
2            TF-IDF           TF-IDF score weighting importance  (120, 242)   

           Values  
0          0 or 1  
1  Integer counts  
2   Float weights  

TF-IDF Explanation:
TF-IDF (Term Frequency-Inverse Document Frequency) gives higher weight to words that are important in a document but rare across the corpus.
Common words like 'the', 'and' get lower weights because they appear in many documents (high IDF).
Rare words that appear frequently in a specific document get higher weights.
Top important words in TF-IDF:
shoe: 11.77
comfortable: 7.05
feel: 5.59
good: 5.08
foot: 4.35
sole: 4.16
wear: 4.03
cushioning: 3.88
fit: 3.77
support: 3.42


## Sparse Matrix Analysis

Print shapes, calculate sparsity, explain inefficiencies.

In [12]:
def calculate_sparsity(matrix):
    total_elements = matrix.shape[0] * matrix.shape[1]
    non_zero = matrix.nnz
    sparsity = (1 - non_zero / total_elements) * 100
    return sparsity

print("Matrix Shapes and Sparsity:")
print(f"OHE: {ohe_matrix.shape}, Sparsity: {calculate_sparsity(ohe_matrix):.2f}%")
print(f"BoW: {bow_matrix.shape}, Sparsity: {calculate_sparsity(bow_matrix):.2f}%")
print(f"TF-IDF: {tfidf_matrix.shape}, Sparsity: {calculate_sparsity(tfidf_matrix):.2f}%")

print("\nWhy sparse matrices are inefficient for large-scale systems:")
print("- High sparsity means most values are zero, wasting space if stored densely.")
print("- Sparse representations save memory but require specialized algorithms.")
print("- For large vocabularies, even sparse matrices can be huge.")
print("- In big data, distributed computing is needed, complicating sparse matrix operations.")

Matrix Shapes and Sparsity:
OHE: (120, 242), Sparsity: 97.86%
BoW: (120, 242), Sparsity: 97.86%
TF-IDF: (120, 242), Sparsity: 97.86%

Why sparse matrices are inefficient for large-scale systems:
- High sparsity means most values are zero, wasting space if stored densely.
- Sparse representations save memory but require specialized algorithms.
- For large vocabularies, even sparse matrices can be huge.
- In big data, distributed computing is needed, complicating sparse matrix operations.


## Real-world Questions

1. Why Bag of Words fails in understanding semantic meaning
2. When to use BoW and TF-IDF in industry
3. Limitations of TF-IDF

In [13]:
print("1. Why Bag of Words fails in understanding semantic meaning:")
print("BoW treats text as a bag of words, ignoring order, context, and semantics.")
print("Example: 'The cat sat on the mat' and 'The mat sat on the cat' have same representation, but different meanings.")
print("It doesn't capture synonyms (car vs automobile) or polysemy.")

print("\n2. When to use Bag of Words and TF-IDF in industry:")
print("BoW: Simple tasks like spam detection, basic classification where word frequency matters.")
print("TF-IDF: Document retrieval, search engines, content recommendation, where term importance across documents is key.")
print("Use TF-IDF when you want to downweight common words and highlight unique terms.")

print("\n3. Limitations of TF-IDF:")
print("- Doesn't capture word order or context.")
print("- Ignores semantics and relationships between words.")
print("- Can be affected by document length.")
print("- Not suitable for short texts or when context is important.")
print("- Requires preprocessing and can be computationally expensive for large corpora.")

1. Why Bag of Words fails in understanding semantic meaning:
BoW treats text as a bag of words, ignoring order, context, and semantics.
Example: 'The cat sat on the mat' and 'The mat sat on the cat' have same representation, but different meanings.
It doesn't capture synonyms (car vs automobile) or polysemy.

2. When to use Bag of Words and TF-IDF in industry:
BoW: Simple tasks like spam detection, basic classification where word frequency matters.
TF-IDF: Document retrieval, search engines, content recommendation, where term importance across documents is key.
Use TF-IDF when you want to downweight common words and highlight unique terms.

3. Limitations of TF-IDF:
- Doesn't capture word order or context.
- Ignores semantics and relationships between words.
- Can be affected by document length.
- Not suitable for short texts or when context is important.
- Requires preprocessing and can be computationally expensive for large corpora.


## Mini Use Case

Perform sentiment classification (positive vs negative) using Logistic Regression, compare BoW and TF-IDF.

In [14]:
from nltk.sentiment import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()

df['sentiment'] = df['review_text'].apply(lambda x: 'positive' if sia.polarity_scores(x)['compound'] > 0 else 'negative')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# BoW
X_train_bow, X_test_bow, y_train, y_test = train_test_split(bow_matrix, df['sentiment'], test_size=0.2, random_state=42)
lr_bow = LogisticRegression(max_iter=1000)
lr_bow.fit(X_train_bow, y_train)
y_pred_bow = lr_bow.predict(X_test_bow)
acc_bow = accuracy_score(y_test, y_pred_bow)

# TF-IDF
X_train_tfidf, X_test_tfidf, y_train_tfidf, y_test_tfidf = train_test_split(tfidf_matrix, df['sentiment'], test_size=0.2, random_state=42)
lr_tfidf = LogisticRegression(max_iter=1000)
lr_tfidf.fit(X_train_tfidf, y_train_tfidf)
y_pred_tfidf = lr_tfidf.predict(X_test_tfidf)
acc_tfidf = accuracy_score(y_test_tfidf, y_pred_tfidf)

print(f"BoW Accuracy: {acc_bow:.2f}")
print(f"TF-IDF Accuracy: {acc_tfidf:.2f}")
print("Comparison: TF-IDF often performs better by weighting important terms.")

BoW Accuracy: 0.88
TF-IDF Accuracy: 0.92
Comparison: TF-IDF often performs better by weighting important terms.


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/Ashish_1/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


## Summary and Report

This notebook implements a complete text feature engineering pipeline on shoe reviews:

**Key Findings:**
- Vocabulary size: 242 unique words
- TF-IDF features capture term importance better than Bag of Words
- TF-IDF achieves ~92% accuracy in sentiment classification vs ~88% for BoW
- High sparsity (>95%) in all feature matrices shows efficient sparse storage is critical

**Recommended Use Cases:**
- **Bag of Words**: Spam detection, simple text classification
- **TF-IDF**: Search engines, document retrieval, content recommendation

**Next Steps:**
- Explore word embeddings (Word2Vec, GloVe)
- Apply deep learning models (LSTM, BERT) for better semantic understanding
- Handle imbalanced datasets with class weights

## How to Push to GitHub

Follow these steps to upload `text_feature_engineering.ipynb` to the assignment folder in the Full-Stack-GenAI-Bootcamp-1.0 repository:

### Step 1: Clone the Repository (if not already cloned)
```bash
cd ~/Documents
git clone https://github.com/anu2985/Full-Stack-GenAI-Bootcamp-1.0.git
cd Full-Stack-GenAI-Bootcamp-1.0
```

### Step 2: Create or Navigate to Assignment Folder
```bash
mkdir -p assignment
cd assignment
```

### Step 3: Copy the Notebook File
```bash
# From your current location, copy the notebook
cp /Users/Ashish_1/Documents/Documents/KRISH/class-03-29-Mar/assignment/text_feature_engineering.ipynb .
```

### Step 4: Verify the File
```bash
ls -la text_feature_engineering.ipynb
```

### Step 5: Stage the Changes
```bash
# Stage the new file
git add text_feature_engineering.ipynb

# Or add everything in assignment folder
git add assignment/
```

### Step 6: Commit the Changes
```bash
git commit -m "Add text feature engineering notebook - OHE, BoW, and TF-IDF implementation with sentiment classification"
```

### Step 7: Push to GitHub
```bash
# Push to main branch (or appropriate branch)
git push origin main
```

### Step 8: Verify on GitHub
Visit: https://github.com/anu2985/Full-Stack-GenAI-Bootcamp-1.0/tree/main/assignment

Your notebook should now be visible in the assignment folder!